<div align="center">

# Convert edf to hdf5

</div>

In [1]:
import os
import numpy as np
import mne
import h5py
import re

In [2]:
def parse_chbmit_summary(summary_path):
    seizures_dict = {}

    with open(summary_path, "r", encoding="latin1") as f:
        text = f.read()

    # split into file sections
    blocks = text.split("File Name:")

    for block in blocks:
        if ".edf" not in block:
            continue

        # ---------------------------------------------------
        # FIX 1: flexible EDF filename matching
        # handles:
        # chb02_16.edf
        # chb02_16+.edf
        # chb02_16_1.edf (future-proof)
        # ---------------------------------------------------
        file_match = re.search(r"(chb\d+_\d+\+?\.edf)", block)

        # fallback: ultra-flexible (safest option)
        if not file_match:
            file_match = re.search(r"(chb\d+_\d+[^\s]*\.edf)", block)

        if not file_match:
            continue

        file_name = file_match.group(1)

        seizures = []

        # ---------------------------------------------------
        # CASE 1: numbered seizures
        # ---------------------------------------------------
        starts_num = re.findall(r"Seizure\s*\d+\s*Start Time:\s*(\d+)", block)
        ends_num   = re.findall(r"Seizure\s*\d+\s*End Time:\s*(\d+)", block)

        if len(starts_num) > 0 and len(ends_num) > 0:
            seizures = [(int(s), int(e)) for s, e in zip(starts_num, ends_num)]

        else:
            # ---------------------------------------------------
            # CASE 2: unnumbered format
            # ---------------------------------------------------
            start = re.search(r"Seizure Start Time:\s*(\d+)", block)
            end   = re.search(r"Seizure End Time:\s*(\d+)", block)

            if start and end:
                seizures = [(int(start.group(1)), int(end.group(1)))]

        seizures_dict[file_name] = seizures

    return seizures_dict





def convert_dataset(seizure_map):
    for file_name, seizures_sec in seizure_map.items():

        edf_path = os.path.join(DATASET_PATH, file_name)
        if not os.path.exists(edf_path):
            print(f"Missing: {edf_path}")
            continue

        print(f"Processing {file_name}")

        # -------------------------
        # Load EDF
        # -------------------------
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw.rename_channels(lambda x: x.strip())

        data = raw.get_data().astype(np.float32)   # (channels, samples)
        fs = float(raw.info["sfreq"])
        ch_names = np.array(raw.ch_names, dtype="S")

        # -------------------------
        # Convert seizures (sec → samples)
        # -------------------------
        seizures_samples = np.array(
            [(int(s * fs), int(e * fs)) for s, e in seizures_sec],
            dtype=np.int64
        )

        if seizures_samples.size == 0:
            seizures_samples = np.zeros((0, 2), dtype=np.int64)

        # -------------------------
        # Save HDF5
        # -------------------------
        h5_path = os.path.join(
            OUTPUT_PATH,
            file_name.replace(".edf", ".h5")
        )

        with h5py.File(h5_path, "w") as f:
            f.create_dataset("signals", data=data, compression="gzip")
            f.create_dataset("sampling_rate", data=fs)
            f.create_dataset("channel_names", data=ch_names)
            f.create_dataset("seizure_intervals", data=seizures_samples)

        print(f"Saved → {h5_path} | seizures: {len(seizures_samples)}")



In [3]:
nums = [f"{i:02d}" for i in range(15, 25)]

In [4]:

for numero in nums:
    texto = f"../Dataset/chb{numero}/chb{numero}-summary.txt"
    seizure_map = parse_chbmit_summary(texto)
    print(seizure_map)



    DATASET_PATH = f"../Dataset/chb{numero}"
    OUTPUT_PATH = f"../Dataset_hdf5/chb{numero}"

    os.makedirs(OUTPUT_PATH, exist_ok=True)



    # RUN
    convert_dataset(seizure_map)



{'chb15_01.edf': [], 'chb15_02.edf': [], 'chb15_03.edf': [], 'chb15_04.edf': [], 'chb15_05.edf': [], 'chb15_06.edf': [(272, 397)], 'chb15_07.edf': [], 'chb15_08.edf': [], 'chb15_09.edf': [], 'chb15_10.edf': [(1082, 1113)], 'chb15_11.edf': [], 'chb15_12.edf': [], 'chb15_13.edf': [], 'chb15_14.edf': [], 'chb15_15.edf': [(1591, 1748)], 'chb15_16.edf': [], 'chb15_17.edf': [(1925, 1960)], 'chb15_19.edf': [], 'chb15_20.edf': [(607, 662)], 'chb15_22.edf': [(760, 965)], 'chb15_26.edf': [], 'chb15_28.edf': [(876, 1066)], 'chb15_29.edf': [], 'chb15_30.edf': [], 'chb15_31.edf': [(1751, 1871)], 'chb15_32.edf': [], 'chb15_33.edf': [], 'chb15_35.edf': [], 'chb15_37.edf': [], 'chb15_40.edf': [(834, 894), (2378, 2497), (3362, 3425)], 'chb15_45.edf': [], 'chb15_46.edf': [(3322, 3429)], 'chb15_49.edf': [(1108, 1248)], 'chb15_50.edf': [], 'chb15_51.edf': [], 'chb15_52.edf': [(778, 849)], 'chb15_54.edf': [(263, 318), (843, 1020), (1524, 1595), (2179, 2250), (3428, 3460)], 'chb15_61.edf': [], 'chb15_62.edf

C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'-'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb15\chb15_01.h5 | seizures: 0
Processing chb15_02.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4, --5
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb15\chb15_02.h5 | seizures: 0
Processing chb15_03.edf


...

Saved → ../Dataset_hdf5/chb24\chb24_15.h5 | seizures: 1
Processing chb24_17.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb24\chb24_17.h5 | seizures: 1
Processing chb24_21.edf


C:\Users\ritaw\AppData\Local\Temp\ipykernel_17008\456356903.py:74: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)


Saved → ../Dataset_hdf5/chb24\chb24_21.h5 | seizures: 1
